# Train & Fine-Tune Road Following Model (Interactive `ipywidgets` UI)

This notebook loads previously saved image datasets (`road_following_A`, `road_following_B`, etc.), automatically checks if an existing trained PyTorch model (`road_following_model.pth`) exists for **Fine-Tuning**, displays training samples & progress using **`ipywidgets`**, and exports the final updated weights directly into an **ONNX model file** (`road_following_model.onnx`) with `opset_version=11` for high-performance CUDA GPU inference on JetRacer Jetson Nano.

| Step | Description |
|------|-------------|
| 1 | Setup Environment & Load Saved Datasets |
| 2 | Initialize Model & Load Existing Weights (Fine-Tuning) |
| 3 | Start Interactive Training UI (`ipywidgets` + Live Sample Display) |
| 4 | Verify Exported ONNX Model with ONNX Runtime on CUDA GPU |

### 1. Setup Environment & Load Saved Datasets

In [8]:
import os
import sys
import glob
import time
import torch
import torchvision
import cv2
import numpy as np
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

try:
    from jetracer.notebooks.xy_dataset import XYDataset
except ImportError:
    from xy_dataset import XYDataset

try:
    from jetracer.utils import bgr8_to_jpeg
except ImportError:
    from utils import bgr8_to_jpeg

TASK = 'road_following'
CATEGORIES = ['apex']

# Find all saved dataset folders matching 'road_following_*'
dataset_dirs = glob.glob(os.path.join(Path.cwd(), f"{TASK}_*"))
if not dataset_dirs:
    dataset_dirs = glob.glob(os.path.join(parent_dir, "notebooks", f"{TASK}_*"))

print(f"[*] Found {len(dataset_dirs)} dataset directories:")
datasets = []
total_samples = 0
for d in dataset_dirs:
    folder_name = os.path.basename(d)
    ds = XYDataset(d, CATEGORIES, random_hflip=True)
    datasets.append(ds)
    print(f"  [+] Loaded '{folder_name}': {len(ds)} samples")
    total_samples += len(ds)

if len(datasets) == 1:
    dataset = datasets[0]
elif len(datasets) > 1:
    dataset = torch.utils.data.ConcatDataset(datasets)
    dataset.categories = CATEGORIES
else:
    dataset = None

print(f"\n[+] Total combined training samples: {total_samples}")


[*] Found 8 dataset directories:
  [+] Loaded 'road_following_A': 190 samples
  [+] Loaded 'road_following_live.ipynb': 0 samples
  [+] Loaded 'road_following_live_pth.ipynb': 0 samples
  [+] Loaded 'road_following_model.onnx': 0 samples
  [+] Loaded 'road_following_model.pth': 0 samples
  [+] Loaded 'road_following_stanley_onnx.py': 0 samples
  [+] Loaded 'road_following_stanley_onnx_ros.py': 0 samples
  [+] Loaded 'road_following_stanley_pth.py': 0 samples

[+] Total combined training samples: 190


### 2. Initialize Model & Check Existing Weights for Fine-Tuning

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dim = 2 * len(CATEGORIES)  # (x, y) coordinates

pth_save_path  = os.path.join(Path.cwd(), "road_following_model.pth")
onnx_save_path = os.path.join(Path.cwd(), "road_following_model.onnx")

model = torchvision.models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, output_dim)
model = model.to(device)

is_finetuning = os.path.exists(pth_save_path)
if is_finetuning:
    try:
        model.load_state_dict(torch.load(pth_save_path, map_location=device))
        print(f"[+] FINE-TUNING MODE: Loaded existing weights from '{pth_save_path}'")
    except Exception as e:
        print(f"[!] Could not load existing weights ({e}). Initializing fresh weights.")
        is_finetuning = False
else:
    print(f"[*] NEW MODEL MODE: Initialized with ImageNet pretrained weights.")

print(f"[*] PyTorch weights save path: {pth_save_path}")
print(f"[*] ONNX export path         : {onnx_save_path}")


[+] FINE-TUNING MODE: Loaded existing weights from 'd:\JetRacer_AI\jetracer\notebooks\road_following_model.pth'
[*] PyTorch weights save path: d:\JetRacer_AI\jetracer\notebooks\road_following_model.pth
[*] ONNX export path         : d:\JetRacer_AI\jetracer\notebooks\road_following_model.onnx


### 3. Interactive Training UI (`ipywidgets` + Live Sample Display)

In [10]:
import ipywidgets
from IPython.display import display

# Interactive Widgets (Identical library & styling as interactive_regression.ipynb)
epochs_widget     = ipywidgets.IntText(description='epochs', value=10)
batch_size_widget = ipywidgets.IntText(description='batch size', value=8)
loss_widget       = ipywidgets.FloatText(description='loss')
progress_widget   = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')
train_button      = ipywidgets.Button(description='Train & Export ONNX', button_style='warning', icon='play')

# Image Preview Widget (format='jpeg', 224x224)
sample_preview_widget = ipywidgets.Image(
    format='jpeg',
    width=224,
    height=224,
    layout=ipywidgets.Layout(border='2px solid #00ff00', border_radius='4px')
)

status_html_widget = ipywidgets.HTML(
    value=f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 10px; border-radius: 6px; width: 320px;">
        <h4 style="margin: 0 0 6px 0; color: #ffffff;">Training Status</h4>
        <p style="margin: 2px 0;"><b>Mode:</b> {"FINE-TUNING" if is_finetuning else "NEW MODEL"}</p>
        <p style="margin: 2px 0;"><b>Total Samples:</b> {total_samples}</p>
        <p style="margin: 2px 0;"><b>Device:</b> {device}</p>
    </div>
    '''
)

def export_onnx_model():
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    try:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11,
            dynamo=False
        )
    except Exception:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11
        )
    print(f"[+] Successfully exported ONNX model (Opset 11) -> '{onnx_save_path}'")

def start_training(b):
    if dataset is None or total_samples == 0:
        print("[!] ERROR: Dataset is empty! Save samples before training.")
        return

    epochs = epochs_widget.value
    batch_size = batch_size_widget.value
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_button.disabled = True
    model.train()
    start_t = time.time()

    print(f"\n[*] Starting Training for {epochs} Epochs...")

    for epoch in range(epochs):
        processed = 0
        sum_loss = 0.0
        for images, category_idx, xy in train_loader:
            images = images.to(device)
            xy = xy.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = 0.0
            for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2)
            loss /= len(category_idx)

            loss.backward()
            optimizer.step()

            count = len(category_idx.flatten())
            processed += count
            sum_loss += float(loss) * count

            # Update Progress Bar & Loss Display
            progress_widget.value = processed / total_samples
            loss_widget.value = sum_loss / processed

            # Update Live Image Preview Widget with Ground Truth Dot!
            try:
                img_np = images[0].cpu().numpy().transpose(1, 2, 0)
                # Un-normalize ImageNet stats
                mean = np.array([0.485, 0.456, 0.406])
                std = np.array([0.229, 0.224, 0.225])
                img_np = (img_np * std + mean) * 255.0
                img_np = np.clip(img_np, 0, 255).astype(np.uint8)
                img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

                target_x = int(224 * (float(xy[0][0]) / 2.0 + 0.5))
                target_y = int(224 * (float(xy[0][1]) / 2.0 + 0.5))
                cv2.circle(img_bgr, (target_x, target_y), 8, (0, 255, 0), -1)

                sample_preview_widget.value = bgr8_to_jpeg(img_bgr)
            except Exception:
                pass

        print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {loss_widget.value:.4f}")

    elapsed = time.time() - start_t
    print(f"[+] Training finished in {elapsed:.1f}s!")

    # 1. Save PyTorch Model (.pth)
    model.eval()
    torch.save(model.state_dict(), pth_save_path)
    print(f"[+] Saved PyTorch model -> '{pth_save_path}'")

    # 2. Export ONNX Model (.onnx)
    export_onnx_model()
    train_button.disabled = False

train_button.on_click(start_training)

# Assemble Integrated ipywidgets UI (Same style as interactive_regression.ipynb)
train_controls = ipywidgets.VBox([
    epochs_widget,
    batch_size_widget,
    progress_widget,
    loss_widget,
    train_button
])

train_ui = ipywidgets.VBox([
    ipywidgets.HBox([sample_preview_widget, status_html_widget]),
    train_controls
])

display(train_ui)


### 4. Verify Exported ONNX Model with ONNX Runtime on CUDA GPU

In [11]:
import onnxruntime as ort

# Release PyTorch GPU RAM cache before ONNX Session Init
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("[*] Verifying exported ONNX model with ONNX Runtime on CUDA GPU...")
available = ort.get_available_providers()
providers = []
if 'CUDAExecutionProvider' in available:
    cuda_options = {
        'device_id': 0,
        'gpu_mem_limit': 1 * 1024 * 1024 * 1024,
        'arena_extend_strategy': 'kNextPowerOfTwo',
        'cudnn_conv_algo_search': 'DEFAULT'
    }
    providers.append(('CUDAExecutionProvider', cuda_options))
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(onnx_save_path, providers=providers)
    input_info = session.get_inputs()[0]
    output_info = session.get_outputs()[0]
    
    print(f"  [+] Loaded Providers: {session.get_providers()}")
    print(f"  [+] Input Name      : {input_info.name} (Shape: {input_info.shape})")
    print(f"  [+] Output Name     : {output_info.name} (Shape: {output_info.shape})")
    
    # Test dummy inference on GPU
    dummy_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
    outs = session.run([output_info.name], {input_info.name: dummy_np})
    print(f"  [+] Test GPU Output : {outs[0].flatten()}")
    print("\n[+] ONNX GPU Model Verification PASSED 100%! Ready for road_following_live.ipynb!")
except Exception as e:
    print(f"[!] ONNX GPU Verification Exception: {e}")


[*] Verifying exported ONNX model with ONNX Runtime on CUDA GPU...
  [+] Loaded Providers: ['CPUExecutionProvider']
  [+] Input Name      : input_0 (Shape: [1, 3, 224, 224])
  [+] Output Name     : output_0 (Shape: [1, 2])
  [+] Test GPU Output : [0.05412862 0.692639  ]

[+] ONNX GPU Model Verification PASSED 100%! Ready for road_following_live.ipynb!
